# Double Gyre Flow Analysis with KoopSim

## The Double Gyre

The **double gyre** is a canonical benchmark for mixing and transport in fluid dynamics. It models two counter-rotating vortices with a time-dependent perturbation that drives chaotic particle advection.

The velocity field is:

$$\dot{x} = -\pi A \sin(\pi f(x,t)) \cos(\pi y)$$
$$\dot{y} = \pi A \cos(\pi f(x,t)) \sin(\pi y) \cdot \frac{\partial f}{\partial x}$$

where $f(x,t) = \varepsilon \sin(\omega t) x^2 + (1 - 2\varepsilon \sin(\omega t))x$.

The domain is $[0, 2] \times [0, 1]$.

## Why Koopman for Mixing Flows?

Koopman spectral analysis can identify **coherent structures** in time-dependent flows. The eigenvalues and eigenfunctions of the Koopman operator reveal transport barriers and mixing regions -- information that is difficult to extract from direct numerical simulation alone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from koopsim import KoopSim
from koopsim.systems import DoubleGyre
from koopsim.utils.visualization import (
    plot_trajectory_comparison,
    plot_phase_portrait,
    plot_eigenspectrum,
    plot_prediction_error,
)

## 1. Generate Training Data

We sample particle trajectories from multiple initial conditions distributed across the gyre domain.

In [ ]:
system = DoubleGyre(A=0.1, epsilon=0.25, omega=2 * np.pi)
print(f"System: {system.name}, state dimension: {system.state_dim}")

# Distribute initial conditions across the domain
rng = np.random.default_rng(42)
n_ic = 20
initial_conditions = [
    np.array([rng.uniform(0.1, 1.9), rng.uniform(0.1, 0.9)])
    for _ in range(n_ic)
]

dt = 0.05
X_train, Y_train = system.generate_snapshots(
    initial_conditions, dt=dt, n_steps=100
)
print(f"Training data: {X_train.shape[0]} snapshot pairs")

## 2. Train EDMD with RBF Dictionary

For the double gyre, an RBF dictionary with spatially distributed centers captures the nonlinear flow structure better than polynomials.

In [ ]:
sim = KoopSim(method="edmd", rbf_centers=50)
sim.fit(X_train, Y_train, dt=dt)
print(sim)

## 3. Predict Particle Trajectories

We predict the trajectory of a test particle and compare to the ground truth ODE integration.

In [ ]:
# Test particle starting in the left gyre
x0_test = np.array([0.5, 0.5])
n_pred_steps = 150
times = np.arange(n_pred_steps + 1) * dt

traj_true = system.generate_trajectory(x0_test, dt=dt, n_steps=n_pred_steps)
traj_pred = sim.predict_trajectory(x0_test, times)

print(f"Trajectory length: {len(times)} time points over t=[0, {times[-1]:.1f}]")

## 4. Visualize: Phase Portrait (Particle Paths)

In [ ]:
fig = plot_phase_portrait([traj_true, traj_pred])
ax = fig.axes[0]
ax.legend(["Ground truth", "", "Koopman prediction", ""])
ax.set_title("Double Gyre -- Particle Trajectory")
ax.set_xlim(-0.1, 2.1)
ax.set_ylim(-0.1, 1.1)
plt.show()

## 5. Trajectory Comparison (x and y vs Time)

In [ ]:
fig = plot_trajectory_comparison(
    times, traj_true, traj_pred, labels=["x", "y"]
)
plt.show()

## 6. Prediction Error Over Time

For chaotic flows like the double gyre, prediction error grows over time. The Koopman model typically provides accurate short-to-medium term predictions.

In [ ]:
from koopsim.core.validation import ModelValidator

errors = ModelValidator.multi_step_error(
    sim.model, traj_true, dt=dt, n_steps=n_pred_steps
)

fig = plot_prediction_error(times[1:], errors)
ax = fig.axes[0]
ax.set_xlabel("Time")
ax.set_title("Double Gyre -- Multi-step Prediction Error")
plt.show()

print(f"Error at t=1.0: {errors[int(1.0/dt)-1]:.6e}")
print(f"Error at t=5.0: {errors[int(5.0/dt)-1]:.6e}")

## 7. Multiple Particle Predictions

Predict and visualize several particles to see how the Koopman model captures the overall flow structure.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

test_ics = [
    np.array([0.3, 0.3]),
    np.array([0.8, 0.7]),
    np.array([1.2, 0.5]),
    np.array([1.7, 0.3]),
]

t_vis = np.arange(101) * dt
for ic in test_ics:
    true = system.generate_trajectory(ic, dt=dt, n_steps=100)
    pred = sim.predict_trajectory(ic, t_vis)
    ax1.plot(true[:, 0], true[:, 1], "-", linewidth=1.2)
    ax1.plot(ic[0], ic[1], "o", markersize=6)
    ax2.plot(pred[:, 0], pred[:, 1], "--", linewidth=1.2)
    ax2.plot(ic[0], ic[1], "o", markersize=6)

for ax, title in zip([ax1, ax2], ["Ground Truth", "Koopman Prediction"]):
    ax.set_xlim(-0.1, 2.1)
    ax.set_ylim(-0.1, 1.1)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Eigenspectrum Analysis

The Koopman eigenspectrum encodes the system's oscillatory content. Eigenvalues near the unit circle correspond to persistent dynamics; those inside correspond to transient/decaying modes.

In [ ]:
fig = plot_eigenspectrum(sim.model)
plt.show()

In [ ]:
spectral = sim.spectral_analysis()

print("Top 5 dominant Koopman modes:")
print(f"{'Mode':>5} {'|lambda|':>10} {'Freq (rad/s)':>14} {'Growth rate':>12}")
print("-" * 45)
for i in spectral["dominant_mode_indices"][:5]:
    print(
        f"{i:5d} {abs(spectral['eigenvalues'][i]):10.4f}"
        f" {spectral['frequencies'][i]:14.4f}"
        f" {spectral['growth_rates'][i]:12.4f}"
    )

## Summary

In this notebook we:
1. Simulated particle trajectories in the time-dependent double gyre
2. Trained an EDMD model with RBF dictionary (50 centers)
3. Predicted particle paths and compared to ground truth
4. Analyzed prediction error growth over time (characteristic of chaotic advection)
5. Examined the Koopman eigenspectrum for oscillatory structure

The double gyre is a challenging benchmark due to its time-dependent and mixing nature. The Koopman approach provides useful short-to-medium term predictions and spectral insight into the flow's coherent structures.